# **Prompting LLMs**

**Author:** Louis G. Binwag III

**Reference:** Course Notes (.ipynb)

---

**Instructions:** Evaluate the models on a subset of the BelebeleLinks to an external site. benchmark, specifically the Filipino subset. This task will require you to work with Filipino prompts.


1. **Load the dataset:** The Belebele dataset is available on HuggingFaceLinks to an external site.. You will need to load the tgl_Latn split (all 900 rows).

2. **Create Filipino prompts:** For each example in the dataset, you will need to construct a multiple-choice question in Filipino. The dataset contains a context, a question, and four possible answers.

3. **Evaluate the models:** Use the gemma3:1b, llama3.2:1b, and the quantized aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k models to answer the questions.

4. **Save the results:** Your output should be one JSONL results file for each model you test (e.g., belebele_results_gemma3:1b.jsonl). Each line in the file should be a JSON object containing the model_name, prompt, response, correct_answer, and whether the model's answer was correct.

**Expected Output:** Jupyter Notebook

In [1]:
!pip install --quiet datasets

In [2]:
from datasets import load_dataset

ds = load_dataset("facebook/belebele", "tgl_Latn")
split = 'validation' if 'validation' in ds else 'test' if 'test' in ds else 'train'

/Users/uwie/Documents/_Github/CSCI161-SocialComputing/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
df = ds[split].to_pandas()
df.head()

,link,question_number,flores_passage,question,mc_answer1,mc_answer2,mc_answer3,mc_answer4,correct_answer_num,dialect,ds
0,https://en.wikibooks.org/wiki/Accordion/Right_...,1,Siguruhing kalmado ang iyong kamay hangga't po...,"Ayon sa sipi, ano ang hindi maituturing na tum...","Para sa mas malakas na volume, dagdagan ang pu...","Hanggang maaari, iwasan ang hindi kinakailanga...",Maging maingat sa pag-abot ng nota habang nana...,Mas bilisan pa ang paggalaw ng mga bellow upan...,1,tgl_Latn,2023-06-01
1,https://en.wikibooks.org/wiki/Accordion/Right_...,2,Siguruhing kalmado ang iyong kamay hangga't po...,"Kapag nagpapatugtog ng akordyon, alin sa mga s...",Mas mabilis na paggalaw,Mas malakas na puwersa,Mas mababang presyon,Mas kakaunting paggalaw ng daliri,1,tgl_Latn,2023-06-01
2,https://en.wikibooks.org/wiki/All_About_Conver...,1,Isa sa mga pinakakaraniwang problema kapag sin...,Bakit putol ang mga border sa mga imahe sa tel...,Upang makapaglagay ng subtitles,Upang mapuno ang buong screen ng imahe,Upang mapasimple ang pag-convert sa ibang format,Upang maging napakalapit ng subtitle sa ibaban...,2,tgl_Latn,2023-06-01
3,https://en.wikibooks.org/wiki/All_About_Conver...,2,Isa sa mga pinakakaraniwang problema kapag sin...,"Ayon sa sipi, alin sa mga sumusunod na problem...",Imahe na hindi mapupuno ang buong screen,Subtitles na bahagyang naputol,Image na mapupuno ang buong screen,Naputol na mga border,2,tgl_Latn,2023-06-01
4,https://en.wikibooks.org/wiki/American_Revolut...,2,Umasa ang plano ng Amerika sa paglulunsad ng m...,Saan may matatagpuan na Britanong garison?,Sapa ng Assunpink,Trenton,Bordentown,Princeton,3,tgl_Latn,2023-06-01


In [4]:
def build_prompt(item):
    """Builds a Filipino multiple-choice question prompt."""
    passage = item["flores_passage"]
    question = item["question"]
    a1 = item["mc_answer1"]
    a2 = item["mc_answer2"]
    a3 = item["mc_answer3"]
    a4 = item["mc_answer4"]

    prompt = (
        "Basahin ang sipi at sagutin ang tanong.\n\n"
        f"Flores Passage: {passage}\n\n"
        f"Question: {question}\n\n"
        "Choices:\n"
        f"(A) {a1}\n"
        f"(B) {a2}\n"
        f"(C) {a3}\n"
        f"(D) {a4}\n\n"
        "Ang tamang sagot ay:"
    )

    return prompt


In [5]:
def clean_response(text):
    """
    Extracts only the first A/B/C/D from Gemma’s response.
    """
    match = re.search(r"[ABCD]", text.upper())
    return f"({match.group(0)})" if match else None

In [6]:
# !curl -fsSL https://ollama.com/install.sh | sh


In [7]:
# import requests

# def query_ollama(model, prompt):
#     """Send a prompt to a locally running Ollama model and return its text response."""
#     response = requests.post(
#         "http://localhost:11434/api/generate",
#         json={
#             "model": model,
#             "prompt": prompt,
#             "stream": False
#         },
#         timeout=60
#     )
#     response.raise_for_status()
#     return response.json()["response"].strip()


In [8]:
# !wsl --install
# !curl -fsSL https://ollama.com/install.sh | sh

import os
import asyncio

# Set LD_LIBRARY_PATH so the system can find ollama's shared libraries
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/x86_64-linux-gnu'

async def run_ollama():
    proc = await asyncio.create_subprocess_shell(
        'ollama serve',
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )
    # The server is now running in the background
    print("Ollama server started.")
    # We don't await proc.communicate() here to let it run in the background

# Start the server
await run_ollama()

# Give the server a moment to start
!sleep 5

Ollama server started.


In [9]:
!ollama pull gemma3:1b
!ollama pull llama3.2:1b
!ollama pull aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 7cd4618c1faf: 100% ▕██████████████████▏ 815 MB                         
pulling e0a42594d802: 100% ▕██████████████████▏  358 B                         
pulling dd084c7d92a3: 100% ▕██████████████████▏ 8.4 KB                         
pulling 3116c5225075: 100% ▕██████████████████▏   77 B                         
pulling 120007c81bf8: 100% ▕██████████████████▏  492 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 

In [10]:
!ollama list

NAME                                        ID              SIZE      MODIFIED               
aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k    341953000c44    3.8 GB    Less than a second ago    
llama3.2:1b                                 baf6a787fdff    1.3 GB    2 seconds ago             
gemma3:1b                                   8648f39daa8f    815 MB    5 seconds ago             


In [11]:
import subprocess
import re


In [12]:
def ask_ollama(prompt, model="gemma3:1b"):
    """
    Sends the prompt to the Ollama model and returns raw text output.
    """
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    return result.stdout.decode("utf-8").strip()

In [13]:
sample_item = df.iloc[0]
prompt = build_prompt(sample_item)

print(ask_ollama(prompt, "gemma3:1b"))
print(ask_ollama(prompt, "llama3.2:1b"))
print(ask_ollama(prompt, "aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k"))



Ang tamang sagot ay **(B) Hanggang maaari, iwasan ang hindi kinakailangang paggalaw upang mapreserba ang iyong stamina.**

Narito ang paliwanag:

Ang sipi ay nagpapahiwatig na mahalaga ang pagiging kalmado at pag-iwas sa labis na paggalaw. Ito ay naglalayong mag-preserve ng stamina ng practitioner.  Ang pagpapataas ng volume ay hindi magiging epektibo kung hindi maayos ang tamang pagpindot.
Sa sipi, ang hindi maituturing na tumpak na paalala para sa matagumpay na pagpapatugtog ng akordyon ay:

C. Maging maingat sa pag-abot ng nota habang nananatiling kalmado ang kamay
Ang tamang sagot ay **(A) Para sa mas malakas na volume, dagdagan ang puwersa ng pagpindot sa teklado**. 

Dahil sa sipi ay nagpapaalala na hindi katulad ng piyano ang akordyon na kailangan pindutin nang madiin ang mga teklado para lumakas ang tunog. Sa halip, dapat gamitin ang presyon at bilis ng paggalaw ng mga bellow para magdagdag ng volume.


In [ ]:
import json
import re
import subprocess
from tqdm import tqdm

def ask_ollama(prompt, model="gemma3:1b"):
    """
    Sends the prompt to the specified Ollama model and returns raw text output.
    """
    try:
        result = subprocess.run(
            ["ollama", "run", model],
            input=prompt.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=120
        )
        return result.stdout.decode("utf-8").strip()
    except subprocess.TimeoutExpired:
        return "[ERROR: Timeout]"
    except Exception as e:
        return f"[ERROR: {e}]"

def clean_response(text):
    """
    Extracts only the first (A/B/C/D) from the model's response.
    """
    match = re.search(r"[ABCD]", text.upper())
    return f"({match.group(0)})" if match else None

def map_correct_answer(num):
    """
    Converts correct_answer_num (1–4) to corresponding letter.
    """
    mapping = {1: "(A)", 2: "(B)", 3: "(C)", 4: "(D)"}
    return mapping.get(num, None)

def evaluate_model(df, model_name):
    """
    Evaluates a model on the Belebele Filipino subset and saves JSONL results.
    Output JSONL contains only:
        model_name, prompt, response, correct_answer (1–4), is_correct
    """
    results = []

    for _, item in tqdm(df.iterrows(), total=len(df)):
        prompt = build_prompt(item)
        response = ask_ollama(prompt, model=model_name)
        parsed = clean_response(response)

        # Compare model output to mapped correct answer
        correct_letter = map_correct_answer(item["correct_answer_num"])
        is_correct = parsed == correct_letter

        results.append({
            "model_name": model_name,
            "prompt": prompt,
            "response": response,
            "correct_answer": int(item["correct_answer_num"]),  # keep numeric
            "is_correct": is_correct
        })

    # Save to JSONL
    out_file = f"belebele_results_{model_name.replace('/', '_')}.jsonl"
    with open(out_file, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Output to {out_file}")
    return results


In [ ]:
import json
import re
import subprocess
from tqdm import tqdm

def ask_ollama(prompt, model="gemma3:1b"):
    """
    Sends the prompt to the specified Ollama model and returns raw text output.
    """
    try:
        result = subprocess.run(
            ["ollama", "run", model],
            input=prompt.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=120
        )
        return result.stdout.decode("utf-8").strip()
    except subprocess.TimeoutExpired:
        return "[ERROR: Timeout]"
    except Exception as e:
        return f"[ERROR: {e}]"



def clean_response(text):
    """
    Extracts only the first (A/B/C/D) from the model's response.
    """
    match = re.search(r"[ABCD]", text.upper())
    return f"({match.group(0)})" if match else None



def map_correct_answer(num):
    """
    Converts correct_answer_num (1–4) to corresponding letter.
    """
    mapping = {1: "(A)", 2: "(B)", 3: "(C)", 4: "(D)"}
    return mapping.get(num, None)



def evaluate_model(df, model_name):
    """
    Evaluates and outputs:
    
    model_name, prompt, response, correct_answer (1–4), is_correct
    """
    results = []

    for _, item in tqdm(df.iterrows(), total=len(df)):
        prompt = build_prompt(item)
        response = ask_ollama(prompt, model=model_name)
        parsed = clean_response(response)

        # Safely extract the numeric correct answer
        correct_num = (
            item.get("correct_answer_num")
            or item.get("correct_answer_idx")
            or item.get("correct_answer")
        )

        # Convert from numpy or float to int if needed
        if pd.notnull(correct_num):
            correct_num = int(correct_num)
        else:
            correct_num = None

        # Compare to mapped letter form (for correctness check)
        correct_letter = map_correct_answer(correct_num)
        is_correct = parsed == correct_letter if correct_letter else False

        results.append({
            "model_name": model_name,
            "prompt": prompt,
            "response": response,
            "correct_answer": correct_num,  # keep as 1–4
            "is_correct": is_correct
        })

    """
    Save to JSONL
    """

    out_file = f"belebele_results_{model_name.replace('/', '_')}.jsonl"
    with open(out_file, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"Saved {out_file}")
    return results



In [ ]:
test_df = df.head(5)   

print(test_df = evaluate_model(df, "gemma3:1b"))
print(test_df = evaluate_model(df, "llama3.2:1b"))
print(test_df = evaluate_model(df, "aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k"))


  0%|          | 3/900 [00:37<2:45:46, 11.09s/it]

In [17]:
results_gemma = evaluate_model(df, "gemma3:1b")
results_llama = evaluate_model(df, "llama3.2:1b")
results_sealion = evaluate_model(df, "aisingapore/Gemma-SEA-LION-v3-9B-IT:q2_k")

  5%|▌         | 48/900 [10:12<3:01:18, 12.77s/it]


KeyboardInterrupt: 